# 02 — Data Preprocessing

**AttriSense · Employee Attrition Prediction & Analytics System**

---

## Introduction

This notebook implements the full preprocessing pipeline for the IBM HR Analytics Employee Attrition dataset. Every transformation is explicit, justified, and logged — nothing is applied silently.

**Scope of this notebook:**

| Step | Action |
|------|--------|
| Feature inspection | Review all 35 raw columns individually |
| Duplicate handling | Detect and remove duplicate rows/IDs |
| Missing values | Inspect and document (impute only if needed) |
| Column removal | Drop only columns with proven zero variance |
| Outlier analysis | IQR diagnostic; capping only where justified |
| Encoding | One-hot for nominal; integer for ordinal |
| Scaling | Applied only when required (off by default for tree models) |
| Persistence | Save cleaned + preprocessed datasets and transformer |

**Prerequisite:** `01_Data_Understanding.ipynb`

**Outputs:**
- `data/processed/employee_attrition_cleaned.parquet`
- `data/processed/employee_attrition_preprocessed.parquet`
- `models/feature_transformer.joblib`

## Library Imports

Reusable logic lives in `src/attrisense/data/preprocessing.py`. The notebook calls those functions and displays every intermediate result.

In [1]:
import pandas as pd

from attrisense.config import load_config
from attrisense.data import (
    feature_registry_dataframe,
    handle_duplicates,
    inspect_missing_values,
    inspect_outliers_iqr,
    load_raw_data,
    run_preprocessing_pipeline,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", "{:.2f}".format)

config = load_config()
df = load_raw_data()

print(f"Loaded raw data: {df.shape[0]:,} rows × {df.shape[1]} columns")

Loaded raw data: 1,470 rows × 35 columns


## Step 1 — Inspect Every Feature

Each of the 35 raw columns is reviewed below. The `action` and `justification` columns document the preprocessing decision before any code runs.

**Decision categories:**
- **Keep** — column enters the feature set as-is
- **Keep as integer** — ordinal scale with natural order
- **Keep; one-hot encode** — nominal category without order
- **Keep; encode to 0/1** — target variable
- **Keep in cleaned data; exclude from feature matrix** — identifier
- **Remove** — zero variance or no predictive value

In [2]:
registry = feature_registry_dataframe()
registry

,column,kind,description,action,justification
0,Age,continuous,Employee age in years,Keep,Valid range 18–60 in dataset; used as continuo...
1,Attrition,target,Whether the employee left the company,Keep; encode to 0/1,Binary target variable for classification.
2,BusinessTravel,nominal,"Travel frequency: Non-Travel, Travel_Rarely, T...",Keep; one-hot encode,Nominal category with 3 levels; no natural ord...
3,DailyRate,continuous,Daily rate of pay,Keep,Numeric compensation field; scale only for lin...
4,Department,nominal,Department assignment,Keep; one-hot encode,"Nominal with 3 levels (HR, R&D, Sales)."
5,DistanceFromHome,continuous,Miles from home to workplace,Keep,Integer distance 1–29; valid commute proxy.
6,Education,ordinal,Education level coded 1–5,Keep as integer,Ordered scale (Below College through Doctor); ...
7,EducationField,nominal,Field of study,Keep; one-hot encode,Nominal with 6 categories; no inherent order.
8,EmployeeCount,constant,Count of employees (always 1 per row),Remove,"Single unique value (1); zero variance, no pre..."
9,EmployeeNumber,identifier,Unique employee identifier,Keep in cleaned data; exclude from feature matrix,Primary key for traceability; not a causal pre...


### Verify registry against actual data

Cross-check documented decisions with observed unique values and dtypes from the loaded dataset.

In [3]:
verification = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "unique_values": [df[c].nunique() for c in df.columns],
    "sample_values": [
        str(sorted(df[c].unique()[:5])) if df[c].nunique() <= 10
        else f"[{df[c].min()}, ..., {df[c].max()}]"
        if pd.api.types.is_numeric_dtype(df[c])
        else str(df[c].value_counts().head(2).to_dict())
        for c in df.columns
    ],
})
verification

,column,dtype,unique_values,sample_values
0,Age,int64,43,"[18, ..., 60]"
1,Attrition,str,2,"['No', 'Yes']"
2,BusinessTravel,str,3,"['Non-Travel', 'Travel_Frequently', 'Travel_Ra..."
3,DailyRate,int64,886,"[102, ..., 1499]"
4,Department,str,3,"['Human Resources', 'Research & Development', ..."
5,DistanceFromHome,int64,29,"[1, ..., 29]"
6,Education,int64,5,"[np.int64(1), np.int64(2), np.int64(3), np.int..."
7,EducationField,str,6,"['Life Sciences', 'Marketing', 'Medical', 'Oth..."
8,EmployeeCount,int64,1,[np.int64(1)]
9,EmployeeNumber,int64,1470,"[1, ..., 2068]"


## Step 2 — Handle Duplicate Rows

Duplicate records would inflate sample counts and could leak across train/test splits. We check both full-row duplicates and repeated `EmployeeNumber` values.

**Expected result (from data understanding):** 0 duplicates. The handler still runs so the pipeline is safe if the source file changes.

In [4]:
deduped, dup_report = handle_duplicates(df, config.id_column)

print(f"Duplicate rows before     : {dup_report.duplicate_rows}")
print(f"Duplicate EmployeeNumber  : {dup_report.duplicate_ids}")
print(f"Rows removed              : {dup_report.rows_removed}")
print(f"Shape after deduplication : {deduped.shape}")
print(f"Action                    : {dup_report.action_taken}")

Duplicate rows before     : 0
Duplicate EmployeeNumber  : 0
Rows removed              : 0
Shape after deduplication : (1470, 35)
Action                    : No duplicates found; no rows removed.


## Step 3 — Inspect Missing Values

Missing data requires an imputation strategy. We count nulls per column before deciding whether to impute, drop rows, or drop columns.

**Decision rule:** Impute only when missing values exist. This dataset was verified in the data understanding notebook to have complete records.

In [5]:
missing_report = inspect_missing_values(deduped)

print(f"Total missing values : {missing_report.total_missing}")
print(f"Columns affected     : {len(missing_report.columns_with_missing)}")
print(f"Action               : {missing_report.action_taken}")

if missing_report.columns_with_missing:
    print(pd.Series(missing_report.columns_with_missing, name="missing_count"))
else:
    print("No imputation applied.")

Total missing values : 0
Columns affected     : 0
Action               : No missing values; imputation not applied.
No imputation applied.


## Step 4 — Remove Irrelevant Columns

Columns are removed **only** when they carry zero variance. Each drop is justified individually — we do not remove columns based on correlation or feature importance at this stage.

| Column | Unique Values | Reason for Removal |
|--------|---------------|-------------------|
| `EmployeeCount` | 1 (always 1) | No variance — every row is a single employee |
| `Over18` | 1 (always "Y") | No variance — all employees are adults |
| `StandardHours` | 1 (always 80) | No variance — identical for all records |

**Not removed:**
- `EmployeeNumber` — unique identifier, kept for traceability but excluded from the model feature matrix
- All other columns — retained for encoding and modeling

In [6]:
constant_cols = [c for c in deduped.columns if deduped[c].nunique() == 1]

constant_detail = pd.DataFrame({
    "column": constant_cols,
    "sole_value": [deduped[c].iloc[0] for c in constant_cols],
    "in_config_drop_list": [c in config.drop_columns for c in constant_cols],
})
print("Constant columns detected in raw data:")
constant_detail

Constant columns detected in raw data:


,column,sole_value,in_config_drop_list
0,EmployeeCount,1,True
1,Over18,Y,True
2,StandardHours,80,True


## Step 5 — Outlier Inspection

We apply the IQR rule (Q1 − 1.5×IQR, Q3 + 1.5×IQR) as a **diagnostic** tool. Flagged values are not automatically removed or capped.

**Why no automatic capping:**
- `MonthlyIncome` — high earners are valid, not errors
- `TotalWorkingYears` / `YearsAtCompany` — senior employees with 30–40 years tenure are real
- `PerformanceRating` — only two values (3, 4); IQR flags most rows as "outliers" because IQR = 0
- `TrainingTimesLastYear` — count variable 0–6; higher counts are valid
- `StockOptionLevel` — level 3 is the maximum benefit tier, not an anomaly

`cap_outliers` is set to `false` in `configs/config.yaml`.

In [7]:
outlier_cols = config.features.continuous + config.features.ordinal
outlier_report = inspect_outliers_iqr(
    deduped,
    outlier_cols,
    multiplier=config.preprocessing.outlier_iqr_multiplier,
)

print(outlier_report.action_taken)
outlier_report.details.sort_values("outlier_count", ascending=False)

Outlier flags recorded for review. No capping or row removal applied — flagged values fall within plausible HR ranges or are discrete counts.


,column,min,max,q1,q3,iqr_lower,iqr_upper,outlier_count,outlier_pct,unique_values
9,TrainingTimesLastYear,0,6,2.00,3.00,0.50,4.50,238,16.19,7
19,PerformanceRating,3,4,3.00,3.00,3.00,3.00,226,15.37,2
4,MonthlyIncome,1009,19999,2911.00,8379.00,-5291.00,16581.00,114,7.76,1349
12,YearsSinceLastPromotion,0,15,0.00,3.00,-4.50,7.50,107,7.28,16
10,YearsAtCompany,0,40,3.00,9.00,-6.00,18.00,104,7.07,37
21,StockOptionLevel,0,3,0.00,1.00,-1.50,2.50,85,5.78,4
8,TotalWorkingYears,0,40,6.00,15.00,-7.50,28.50,63,4.29,40
6,NumCompaniesWorked,0,9,1.00,4.00,-3.50,8.50,52,3.54,10
11,YearsInCurrentRole,0,18,2.00,7.00,-5.50,14.50,21,1.43,19
13,YearsWithCurrManager,0,17,2.00,7.00,-5.50,14.50,14,0.95,18


## Step 6 — Encode Categorical Variables

Encoding strategy depends on whether a column has a natural order.

### Nominal → One-Hot Encoding (`drop='first'`)

| Column | Levels | Rationale |
|--------|--------|-----------|
| `BusinessTravel` | 3 | No order between travel categories |
| `Department` | 3 | Departments are unordered |
| `EducationField` | 6 | Fields of study have no ranking |
| `Gender` | 2 | Unordered categories |
| `JobRole` | 9 | Job titles are not ordinal |
| `MaritalStatus` | 3 | Unordered categories |
| `OverTime` | 2 | Yes/No flag |

Using `drop='first'` avoids the dummy variable trap (multicollinearity).

### Ordinal → Keep as Integer

| Column | Range | Rationale |
|--------|-------|-----------|
| `Education` | 1–5 | Below College → Doctor |
| `EnvironmentSatisfaction` | 1–4 | Likert scale |
| `JobInvolvement` | 1–4 | Likert scale |
| `JobLevel` | 1–5 | Hierarchy level |
| `JobSatisfaction` | 1–4 | Likert scale |
| `PerformanceRating` | 3–4 | Rating scale (limited levels in data) |
| `RelationshipSatisfaction` | 1–4 | Likert scale |
| `StockOptionLevel` | 0–3 | Benefit tier |
| `WorkLifeBalance` | 1–4 | Likert scale |

One-hot encoding ordinal columns would discard the ordering information.

### Target → Binary Integer

| Original | Encoded |
|----------|---------|
| `Yes` | 1 |
| `No` | 0 |

In [8]:
print("Nominal columns (one-hot):")
for col in config.features.nominal:
    print(f"  {col}: {deduped[col].nunique()} levels → {deduped[col].unique().tolist()}")

print("\nOrdinal columns (kept as integer):")
for col in config.features.ordinal:
    print(f"  {col}: {sorted(deduped[col].unique().tolist())}")

print("\nContinuous columns (kept as numeric):")
print(f"  {config.features.continuous}")

Nominal columns (one-hot):
  BusinessTravel: 3 levels → ['Travel_Rarely', 'Travel_Frequently', 'Non-Travel']
  Department: 3 levels → ['Sales', 'Research & Development', 'Human Resources']
  EducationField: 6 levels → ['Life Sciences', 'Other', 'Medical', 'Marketing', 'Technical Degree', 'Human Resources']
  Gender: 2 levels → ['Female', 'Male']
  JobRole: 9 levels → ['Sales Executive', 'Research Scientist', 'Laboratory Technician', 'Manufacturing Director', 'Healthcare Representative', 'Manager', 'Sales Representative', 'Research Director', 'Human Resources']
  MaritalStatus: 3 levels → ['Single', 'Married', 'Divorced']
  OverTime: 2 levels → ['Yes', 'No']

Ordinal columns (kept as integer):
  Education: [1, 2, 3, 4, 5]
  EnvironmentSatisfaction: [1, 2, 3, 4]
  JobInvolvement: [1, 2, 3, 4]
  JobLevel: [1, 2, 3, 4, 5]
  JobSatisfaction: [1, 2, 3, 4]
  PerformanceRating: [3, 4]
  RelationshipSatisfaction: [1, 2, 3, 4]
  StockOptionLevel: [0, 1, 2, 3]
  WorkLifeBalance: [1, 2, 3, 4]

Con

## Step 7 — Scaling Decision

`StandardScaler` (zero mean, unit variance) is needed for distance-based and linear models (Logistic Regression, SVM) but **not** for tree-based models (Random Forest, XGBoost).

| Setting | Value | Reason |
|---------|-------|--------|
| `apply_scaling` | `False` | Default pipeline targets XGBoost; trees are scale-invariant |
| `scale_when_required` | 14 continuous columns | Available when linear models are trained later |

The fitted `ColumnTransformer` is saved to `models/feature_transformer.joblib`. A separate modeling notebook can re-fit with `apply_scaling=True` for linear models.

In [9]:
APPLY_SCALING = False

print(f"Scaling applied in this run: {APPLY_SCALING}")
print(f"Columns configured for scaling when required:")
for col in config.features.scale_when_required:
    print(f"  - {col}")

Scaling applied in this run: False
Columns configured for scaling when required:
  - Age
  - DailyRate
  - DistanceFromHome
  - HourlyRate
  - MonthlyIncome
  - MonthlyRate
  - NumCompaniesWorked
  - PercentSalaryHike
  - TotalWorkingYears
  - TrainingTimesLastYear
  - YearsAtCompany
  - YearsInCurrentRole
  - YearsSinceLastPromotion
  - YearsWithCurrManager


## Step 8 — Run Pipeline and Save Outputs

The full pipeline executes all steps above in sequence and writes three artifacts to disk.

In [10]:
cleaned, preprocessed, report = run_preprocessing_pipeline(
    df,
    config=config,
    apply_scaling=APPLY_SCALING,
    save_artifacts=True,
)

print("=" * 55)
print("PREPROCESSING REPORT")
print("=" * 55)
print(f"Input shape        : {report.input_shape}")
print(f"Cleaned shape      : {report.cleaned_shape}")
print(f"Preprocessed shape : {report.preprocessed_shape}")
print(f"Dropped columns    : {report.dropped_columns}")
print(f"Duplicate action   : {report.duplicate_report.action_taken}")
print(f"Missing action     : {report.missing_report.action_taken}")
print(f"Outlier action     : {report.outlier_report.action_taken}")
print(f"Scaling applied    : {report.scaling_applied}")
print(f"Feature columns    : {report.encoding_summary['output_feature_count']}")
print()
print("Saved artifacts:")
for name, path in report.output_paths.items():
    print(f"  {name}: {path}")

PREPROCESSING REPORT
Input shape        : (1470, 35)
Cleaned shape      : (1470, 32)
Preprocessed shape : (1470, 47)
Dropped columns    : ['EmployeeCount', 'Over18', 'StandardHours']
Duplicate action   : No duplicates found; no rows removed.
Missing action     : No missing values; imputation not applied.
Outlier action     : Outlier flags recorded for review. No capping or row removal applied — flagged values fall within plausible HR ranges or are discrete counts.
Scaling applied    : False
Feature columns    : 44

Saved artifacts:
  cleaned: D:\Fun\AttriSense-Employee-Attrition-Prediction\data\processed\employee_attrition_cleaned.parquet
  preprocessed: D:\Fun\AttriSense-Employee-Attrition-Prediction\data\processed\employee_attrition_preprocessed.parquet
  transformer: D:\Fun\AttriSense-Employee-Attrition-Prediction\models\feature_transformer.joblib


### Cleaned dataset preview

Human-readable dtypes preserved — suitable for EDA and reporting.

In [11]:
cleaned.head(3)

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,2,Female,94,3,2,Sales Executive,4,Single,5993,19479,8,Yes,11,3,1,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,2,3,Male,61,2,2,Research Scientist,2,Married,5130,24907,1,No,23,4,4,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,4,4,Male,92,2,1,Laboratory Technician,3,Single,2090,2396,6,Yes,15,3,2,0,7,3,3,0,0,0,0


### Preprocessed dataset preview

Model-ready matrix: identifier, target (string + label), and encoded features.

In [12]:
feature_cols = [c for c in preprocessed.columns if c not in (
    config.id_column, config.target_column, "Attrition_label"
)]
print(f"Model feature columns ({len(feature_cols)}):")
print(feature_cols)

preprocessed.head(3)

Model feature columns (44):
['BusinessTravel_Travel_Frequently', 'BusinessTravel_Travel_Rarely', 'Department_Research & Development', 'Department_Sales', 'EducationField_Life Sciences', 'EducationField_Marketing', 'EducationField_Medical', 'EducationField_Other', 'EducationField_Technical Degree', 'Gender_Male', 'JobRole_Human Resources', 'JobRole_Laboratory Technician', 'JobRole_Manager', 'JobRole_Manufacturing Director', 'JobRole_Research Director', 'JobRole_Research Scientist', 'JobRole_Sales Executive', 'JobRole_Sales Representative', 'MaritalStatus_Married', 'MaritalStatus_Single', 'OverTime_Yes', 'Education', 'EnvironmentSatisfaction', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'WorkLifeBalance', 'Age', 'DailyRate', 'DistanceFromHome', 'HourlyRate', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'PercentSalaryHike', 'TotalWorkingYears', 'TrainingTimesLastYear', 'YearsAtCompany', 'YearsInCurrentRole'

,EmployeeNumber,Attrition,Attrition_label,BusinessTravel_Travel_Frequently,BusinessTravel_Travel_Rarely,Department_Research & Development,Department_Sales,EducationField_Life Sciences,EducationField_Marketing,EducationField_Medical,EducationField_Other,EducationField_Technical Degree,Gender_Male,JobRole_Human Resources,JobRole_Laboratory Technician,JobRole_Manager,JobRole_Manufacturing Director,JobRole_Research Director,JobRole_Research Scientist,JobRole_Sales Executive,JobRole_Sales Representative,MaritalStatus_Married,MaritalStatus_Single,OverTime_Yes,Education,EnvironmentSatisfaction,JobInvolvement,JobLevel,JobSatisfaction,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,WorkLifeBalance,Age,DailyRate,DistanceFromHome,HourlyRate,MonthlyIncome,MonthlyRate,NumCompaniesWorked,PercentSalaryHike,TotalWorkingYears,TrainingTimesLastYear,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,1,Yes,1,0.00,1.00,0.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,1.00,1.00,2.00,2.00,3.00,2.00,4.00,3.00,1.00,0.00,1.00,41.00,1102.00,1.00,94.00,5993.00,19479.00,8.00,11.00,8.00,0.00,6.00,4.00,0.00,5.00
1,2,No,0,1.00,0.00,1.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,1.00,3.00,2.00,2.00,2.00,4.00,4.00,1.00,3.00,49.00,279.00,8.00,61.00,5130.00,24907.00,1.00,23.00,10.00,3.00,10.00,7.00,1.00,7.00
2,4,Yes,1,0.00,1.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,1.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,1.00,2.00,4.00,2.00,1.00,3.00,3.00,2.00,0.00,3.00,37.00,1373.00,2.00,92.00,2090.00,2396.00,6.00,15.00,7.00,3.00,0.00,0.00,0.00,0.00


## Step 9 — Preprocessing Summary

All decisions applied in this notebook:

In [13]:
summary = pd.DataFrame([
    {"step": "Duplicates", "decision": report.duplicate_report.action_taken},
    {"step": "Missing values", "decision": report.missing_report.action_taken},
    {
        "step": "Column removal",
        "decision": f"Dropped {report.dropped_columns} — zero variance only",
    },
    {"step": "Outliers", "decision": report.outlier_report.action_taken},
    {
        "step": "Nominal encoding",
        "decision": "OneHotEncoder(drop='first') on 7 columns",
    },
    {
        "step": "Ordinal encoding",
        "decision": "Kept as integer on 9 columns",
    },
    {
        "step": "Target encoding",
        "decision": "Yes→1, No→0 stored as Attrition_label",
    },
    {
        "step": "Scaling",
        "decision": (
            "StandardScaler applied to 14 continuous columns"
            if report.scaling_applied
            else "Not applied (tree-model default; scaler config ready for linear models)"
        ),
    },
    {
        "step": "Saved cleaned data",
        "decision": report.output_paths.get("cleaned", "N/A"),
    },
    {
        "step": "Saved preprocessed data",
        "decision": report.output_paths.get("preprocessed", "N/A"),
    },
    {
        "step": "Saved transformer",
        "decision": report.output_paths.get("transformer", "N/A"),
    },
])
summary

,step,decision
0,Duplicates,No duplicates found; no rows removed.
1,Missing values,No missing values; imputation not applied.
2,Column removal,"Dropped ['EmployeeCount', 'Over18', 'StandardH..."
3,Outliers,Outlier flags recorded for review. No capping ...
4,Nominal encoding,OneHotEncoder(drop='first') on 7 columns
5,Ordinal encoding,Kept as integer on 9 columns
6,Target encoding,"Yes→1, No→0 stored as Attrition_label"
7,Scaling,Not applied (tree-model default; scaler config...
8,Saved cleaned data,D:\Fun\AttriSense-Employee-Attrition-Predictio...
9,Saved preprocessed data,D:\Fun\AttriSense-Employee-Attrition-Predictio...


**Next step:** `03_EDA.ipynb` — exploratory analysis on the cleaned dataset, or proceed to feature engineering / model building using `employee_attrition_preprocessed.parquet`.